<a href="https://colab.research.google.com/github/niharnandala/ntsb-safety-review-agent/blob/main/notebooks/02_build_episodes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Build the practice cases (episodes)

In `01_data_checks` we figured out how to get trustworthy answer keys (11,168 reply → NTSB-grade pairs).
Now we turn those pairs into actual practice questions the model can train and get tested on.

One episode = time frozen at one reply:
- **what the model can see:** the recommendation + every letter up to that reply (both sides)
- **hidden answer 1:** NTSB's grade for that reply
- **hidden answer 2:** NTSB's actual letter about it (for teaching letter writing later)

**Runtime:** CPU. We save everything to Google Drive so a Colab disconnect doesn't wipe it.

## Setup: Drive + data
We mount Google Drive and save outputs there. Files in Colab's own storage disappear when the session ends.

In [3]:
from google.colab import drive, files
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/ntsb_agent/data"
import os
os.makedirs(OUT, exist_ok=True)

files.upload()   # pick the CAROL zip again
!unzip -o -q "carol_recommendations_data (1).zip" -d data

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving carol_recommendations_data (1).zip to carol_recommendations_data (1).zip


In [4]:
import json, glob, re, random
from collections import Counter
from datetime import date

with open(glob.glob("data/*.json")[0]) as f:
    recs = json.load(f)
print(len(recs), "recommendations")

10000 recommendations


## Step 1 · Cleaning helpers
Same helpers as the data-checks notebook, plus two new things:
- **boilerplate:** NTSB's standard cover letters ("The NTSB is an independent federal agency…") don't say anything
  about the case, so we drop them from the history.
- **dates:** we keep real dates, but only for our own filtering. The model never sees them.
- **undated letters:** a few letters have no date at all. We can't tell if they're before or after the
  reply we're judging, so keeping them could leak the future. We skip them, same as empty ones.
- **future dates:** 2 letters are dated in the year 2500+ (typos). Anything dated after today is skipped.
- **duplicates:** the audit (`01b`) found 6,500+ letters stored twice in the same thread, sometimes with one
  copy cut short. We keep one copy: the longer one when two letters share a date and one contains the other,
  and the earliest one when the exact same text shows up on different dates.

In [5]:
GRADE = re.compile(r'["“]\s*((?:Open|Closed)\s*[-–—]+\s*[A-Za-z /]+?)\s*[."”,]', re.I)
BOILERPLATE = re.compile(r"is an independent federal agency", re.I)

def text(letter):
    return (letter.get("ResponseSummary") or "").strip()

def from_ntsb(letter):
    # some letters are flagged as NTSB but really start with "-From <org person>"
    return bool(letter.get("IsFromNtsb")) and not text(letter).startswith("-From")

def is_boilerplate(letter):
    # NTSB cover letter with no grade in it = nothing useful for the model
    t = text(letter)
    return from_ntsb(letter) and BOILERPLATE.search(t) and not GRADE.search(t)

TODAY = date.today().isoformat()

def dedupe(letters):
    # longest first (ties: earliest first), then skip anything already covered by a kept letter
    keep = []
    for l in sorted(letters, key=lambda l: (-len(text(l)), l["CorrespondenceDate"])):
        t, d = text(l), l["CorrespondenceDate"][:10]
        same_text = any(t == text(k) for k in keep)
        cut_copy = any(k["CorrespondenceDate"][:10] == d and t in text(k) for k in keep)
        if not (same_text or cut_copy):
            keep.append(l)
    return keep

def letters_in_order(addressee):
    # skip empty letters, cover letters, undated letters and future-dated typos; then remove duplicates
    letters = [l for l in (addressee.get("Correspondence") or [])
               if text(l) and l.get("CorrespondenceDate")
               and l["CorrespondenceDate"][:10] <= TODAY and not is_boilerplate(l)]
    return sorted(dedupe(letters), key=lambda l: l["CorrespondenceDate"])

def to_date(letter):
    return date.fromisoformat(letter["CorrespondenceDate"][:10])

def rec_year(srid):
    # "M-89-073" -> 1989, "A-25-038" -> 2025
    yy = int(srid.split("-")[1])
    return 1900 + yy if yy > 30 else 2000 + yy

## Step 2 · Grade matching (the rules we settled on in 01)
Copied straight from the round 3 rules: find the grade in the sentence that names our case,
skip "old news" recap sentences, latest grade wins, handle ranges + comma lists.
One small addition from the last spot check: "as a result of the earlier response" also counts as a recap.

In [6]:
ANY_ID = re.compile(r"\b[AHIMPR]\s*-?\s*\d{2}\s*-\s*\d{1,3}\b")
RECAP = re.compile(
    r"\b(in (its|our|the board's)\b[^.]{0,60}\bletter|previously classified|had (been )?classified"
    r"|was classified|were classified|earlier response)",
    re.I,
)

def id_parts(srid):
    mode, yy, num = srid.split("-")[:3]
    return mode, yy, int(re.sub(r"\D", "", num) or 0)

def own_id_pattern(srid):
    mode, yy, num = id_parts(srid)
    return re.compile(rf"\b{mode}\s*-?\s*{yy}\s*-\s*0*{num}\b", re.I)

def mentions_rec(passage, srid):
    mode, yy, num = id_parts(srid)
    if own_id_pattern(srid).search(passage):
        return True
    rng = rf"\b{mode}\s*-?\s*{yy}\s*-\s*0*(\d+)\s*(?:through|thru|to)\s*-?\s*(?:{mode}\s*-?\s*{yy}\s*-\s*)?0*(\d+)"
    if any(int(lo) <= num <= int(hi) for lo, hi in re.findall(rng, passage, re.I)):
        return True
    lst = rf"\b{mode}\s*-?\s*{yy}\s*-\s*0*\d+((?:\s*(?:,|and|&)\s*(?:and\s*)?-\s*0*\d+)+)"
    for tail in re.findall(lst, passage, re.I):
        if num in [int(n) for n in re.findall(r"-\s*0*(\d+)", tail)]:
            return True
    return False

def sentence_window(txt, pos):
    ends = [m.end() for m in re.finditer(r"[.!?][\"”']?\s+", txt[:pos])]
    start = ends[-2] if len(ends) >= 2 else 0
    return txt[start:pos]

def other_ids(txt, srid):
    return {i for i in ANY_ID.findall(txt) if not mentions_rec(i, srid)}

def grade_for_rec(ntsb_text, srid):
    fresh = [m for m in GRADE.finditer(ntsb_text)
             if not RECAP.search(sentence_window(ntsb_text, m.start()))]
    ours = [m for m in fresh if mentions_rec(sentence_window(ntsb_text, m.start()), srid)]
    if ours:
        return ours[-1].group(1)
    if len(fresh) == 1 and not other_ids(ntsb_text, srid):
        return fresh[0].group(1)
    return None

def reply_is_about(reply, srid):
    return mentions_rec(reply, srid) or not other_ids(reply, srid)

def to_class(grade):
    g = grade.lower()
    side = "Open" if g.startswith("open") else "Closed"
    if "await" in g:
        return None
    if "supersed" in g or "superced" in g:
        return "Closed – Other"
    if "unacceptable" in g:
        return side + " – Unacceptable"
    if "alternate" in g:
        return side + " – Acceptable Alternate"
    if "acceptable" in g or "exceed" in g:
        return side + " – Acceptable"
    return "Closed – Other" if side == "Closed" else None

## Step 3 · Anonymise
We don't want the model to cheat with dates or IDs (e.g. "old cases are usually closed").
- our recommendation's ID → `CASE-0001` etc., other recommendation IDs → `[OTHER-REC]`
  (an ID like A-93-72 has the year in it)
- 4-digit years in the text → `[YEAR]`
- the "-From Name, Title:" line at the top of org letters → `-From [NAME]:`
- dates → "Day N", counted from **Jan 1 of the year in the ID** (A-93-72 → Jan 1, 1993). The audit showed
  NTSB's original letter is missing in more than half the threads, so "first letter = Day 0" was off by
  months or years. The ID year is only approximate, but much closer. Each letter also gets "days since the
  previous letter", which is exact.

Then we fix the style: 80% of 1990s letters are ALL CAPS, while modern letters aren't. ALL CAPS letters get
turned into normal sentence case (common agency names like NTSB, FAA, USCG stay in caps). This runs after
anonymising, so IDs are already replaced and nothing leaks through lowercase IDs.

Known limit: names written inside the body of a letter ("signed by …") aren't caught by this.

In [7]:
YEAR = re.compile(r"\b(19|20)\d{2}\b")
FROM_LINE = re.compile(r"^-From [^:\n]{1,200}:")

def anonymise(txt, srid, case_id):
    txt = own_id_pattern(srid).sub(case_id, txt)
    txt = ANY_ID.sub("[OTHER-REC]", txt)
    txt = FROM_LINE.sub("-From [NAME]:", txt)
    txt = YEAR.sub("[YEAR]", txt)
    return txt

KEEP_UPPER = ["NTSB", "FAA", "USCG", "NHTSA", "FRA", "FTA", "FHWA", "FMCSA", "PHMSA", "RSPA", "DOT", "NASA",
              "NOAA", "OSHA", "EPA", "AD", "AC", "SB", "NPRM", "ANPRM", "CFR", "SAIB", "U.S.", "CASE-",
              "[OTHER-REC]", "[YEAR]", "[NAME]"]

def is_all_caps(txt):
    letters = [c for c in txt if c.isalpha()]
    return len(letters) >= 20 and sum(c.isupper() for c in letters) / len(letters) > 0.9

def fix_caps(txt):
    if not is_all_caps(txt):
        return txt
    out = txt.lower()
    # capital letter at the start and after every sentence end
    out = re.sub(r"(^\s*|[.!?][\"”']?\s+)([a-z])", lambda m: m.group(1) + m.group(2).upper(), out)
    # put known acronyms and our placeholders back in caps
    for word in KEEP_UPPER:
        out = re.sub(r"(?<![A-Za-z])" + re.escape(word.lower()) + r"(?![a-z])", word, out)
    return out

def clean(txt, srid, case_id):
    return fix_caps(anonymise(txt, srid, case_id))

print(clean('THE BOARD CLASSIFIES A-93-72 "OPEN--ACCEPTABLE RESPONSE." THE FAA ISSUED AD 93-15 IN 1993.',
            "A-93-072", "CASE-0001"))

The board classifies CASE-0001 "open--acceptable response." The FAA issued AD 93-15 in [YEAR].


## Outside info check
From the audit: in ~12% of answers, NTSB uses info that isn't in the reply (meetings, phone calls, attachments,
or documents the reply never names). The grade is still fine for training, but the model shouldn't learn to
write letters that mention things it can't see. So we tag those answers here, and leave them out of
letter-writing training later. It's a rough word-based check.

In [8]:
OUTSIDE = re.compile(r"\b(meeting|telephone|phone call|conversation|discussion with|briefing|enclos\w*|attach\w*)\b", re.I)
DOC = re.compile(r"\b(?:Order|AD|AC|Notice|Bulletin|Circular|SAIB|Directive|Rule)\s+(?:No\.\s*)?[A-Z0-9][\w\-/\.]*\d", re.I)

def uses_outside_info(answer, reply):
    if OUTSIDE.search(answer):
        return True
    docs = {d.lower() for d in DOC.findall(answer)}
    return any(d not in reply.lower() for d in docs)

## Step 4 · Build the episodes
For every trusted pair we save one episode. Everything the model may see goes in normal keys.
Everything it must NOT see (real IDs, real dates, the answer) goes under keys starting with `_`
or under `answer`. The tools in Phase 2 will only ever hand out the visible part.

Every letter gets its own id (`L-0001` …) so the tools can say "read letter L-0007".

One more catch from the first run: sometimes NTSB's "answer" is word-for-word the same as a letter that's
already in the history (the same letter stored twice in the database). That's not a new decision, and the
model would see the answer before being asked. We drop those episodes.

In [9]:
case_ids = {}        # real srid -> CASE-xxxx
per_case = Counter() # how many episodes each case has so far
episodes = []
dup_answers = 0

for r in recs:
    srid = r["Srid"]
    for a in r["Addressees"]:
        letters = letters_in_order(a)
        if not letters:
            continue
        # Day 0 = Jan 1 of the year in the ID (NTSB's original letter is often missing)
        day0 = min(date(rec_year(srid), 1, 1), to_date(letters[0]))
        for i, l in enumerate(letters):
            if from_ntsb(l) or i + 1 >= len(letters) or not from_ntsb(letters[i + 1]):
                continue
            reply, ntsb = text(l), text(letters[i + 1])
            grade = grade_for_rec(ntsb, srid)
            label = to_class(grade) if grade else None
            if not label or not reply_is_about(reply, srid):
                continue

            case_id = case_ids.setdefault(srid, f"CASE-{len(case_ids) + 1:04d}")
            history = []
            for j, h in enumerate(letters[: i + 1]):          # everything up to and including the reply
                history.append({
                    "letter_id": f"L-{j + 1:04d}",
                    "day": (to_date(h) - day0).days,
                    "days_since_previous": (to_date(h) - to_date(letters[j - 1])).days if j > 0 else None,
                    "from": "NTSB" if from_ntsb(h) else "ORG",
                    "type": h.get("CorrespondenceType", ""),
                    "text": clean(text(h), srid, case_id),
                    "_date": h["CorrespondenceDate"][:10],
                })

            answer_text = clean(ntsb, srid, case_id)
            if any(h["text"] == answer_text for h in history):
                dup_answers += 1           # same letter stored twice -> not a new decision
                continue

            per_case[case_id] += 1
            episodes.append({
                "episode_id": f"{case_id}-E{per_case[case_id]}",
                "case_id": case_id,
                "mode": r.get("Mode", ""),
                "priority": r.get("Priority", ""),
                "most_wanted": bool(r.get("MostWanted")),
                "recipient": a.get("AddressAcronym", ""),
                "recommendation": clean(r.get("Subject") or "", srid, case_id),
                "history": history,
                "to_evaluate": history[-1]["letter_id"],
                "answer": {
                    "label": label,
                    "grade_raw": grade,
                    "ntsb_letter": answer_text,
                    "outside_info": uses_outside_info(ntsb, reply),
                },
                "batch": bool(other_ids(ntsb, srid) or other_ids(reply, srid)),
                "_srid": srid,
                "_reply_date": l["CorrespondenceDate"][:10],
                "_answer_date": letters[i + 1]["CorrespondenceDate"][:10],
            })

print(len(episodes), "episodes from", len(per_case), "recommendations")
print(dup_answers, "dropped: answer was already in the history")
print(sum(e["answer"]["outside_info"] for e in episodes), "tagged: NTSB used info that isn't in the reply")
print(Counter(e["answer"]["label"] for e in episodes).most_common())

11175 episodes from 5069 recommendations
0 dropped: answer was already in the history
1243 tagged: NTSB used info that isn't in the reply
[('Open – Acceptable', 4544), ('Closed – Acceptable', 3588), ('Open – Unacceptable', 1005), ('Closed – Unacceptable', 643), ('Closed – Acceptable Alternate', 590), ('Closed – Other', 520), ('Open – Acceptable Alternate', 285)]


## Step 5 · Split by recommendation (picked on purpose)
All episodes of one recommendation go to the same side (train, validation or test), so the model can
never see part of a case in training and then get tested on the rest of it.

A plain random split gave us only 15 recent cases (2010+) in the test set and ~8 examples of the rarest grade.
That's too few to score anything fairly, and recent letters are what an analyst would actually see today.
So we pick the test set on purpose, still by whole recommendation:
1. first, cases with recent replies, until there are ~100 recent test episodes
2. then, cases that cover each grade, until every grade has at least 20
3. then random cases until we reach ~300

Validation gets the same treatment with smaller targets (~50 recent, 15 per grade). Everything else is training.
Cases with more than 10 episodes stay in training, so one huge case can't take over the test set.
We'll say in the write-up that the test set was stratified like this.

In [10]:
random.seed(42)
by_case = {}
for e in episodes:
    e["recent"] = int(e["_reply_date"][:4]) >= 2010
    by_case.setdefault(e["_srid"], []).append(e)

LABELS = sorted({e["answer"]["label"] for e in episodes})
cases = sorted(by_case)
random.shuffle(cases)
small = [c for c in cases if len(by_case[c]) <= 10]   # huge cases stay in train
split_of = {}

def pick(name, size, recent_target, per_label):
    eps = []
    def take(c):
        split_of[c] = name
        eps.extend(by_case[c])
    def free():
        return [c for c in small if c not in split_of]

    # 1. recent cases first
    for c in free():
        if sum(e["recent"] for e in eps) >= recent_target:
            break
        if any(e["recent"] for e in by_case[c]):
            take(c)
    # 2. every grade gets enough examples
    for label in LABELS:
        for c in free():
            if sum(e["answer"]["label"] == label for e in eps) >= per_label:
                break
            if any(e["answer"]["label"] == label for e in by_case[c]):
                take(c)
    # 3. top up with random cases
    for c in free():
        if len(eps) >= size:
            break
        take(c)

pick("test", size=300, recent_target=100, per_label=20)
pick("val", size=300, recent_target=50, per_label=15)

for e in episodes:
    e["split"] = split_of.get(e["_srid"], "train")

print(Counter(e["split"] for e in episodes))
print("recent episodes per split:", Counter(e["split"] for e in episodes if e["recent"]))
print("\ntest set per grade:")
for name, n in Counter(e["answer"]["label"] for e in episodes if e["split"] == "test").most_common():
    print(f"{n:5}  {name}")

Counter({'train': 10574, 'test': 301, 'val': 300})
recent episodes per split: Counter({'train': 131, 'test': 101, 'val': 50})

test set per grade:
  116  Open – Acceptable
   60  Closed – Acceptable
   42  Open – Unacceptable
   23  Closed – Unacceptable
   20  Open – Acceptable Alternate
   20  Closed – Acceptable Alternate
   20  Closed – Other


## Step 6 · Leak test
If any of these fail, we stop and fix it before training. This is the "no peeking" guarantee.
- nothing in the history is dated after the reply we're judging
- the NTSB answer letter is never in the history
- no recommendation shows up in two splits
- no 4-digit year and no real recommendation ID survive in anything the model can see

In [11]:
def visible(e):
    # what the model is allowed to see: no answer, nothing starting with "_"
    v = {k: val for k, val in e.items() if k != "answer" and not k.startswith("_")}
    v["history"] = [{k: val for k, val in h.items() if not k.startswith("_")} for h in e["history"]]
    return v

problems = []
splits_per_case = {}
for e in episodes:
    for h in e["history"]:
        if h["_date"] > e["_reply_date"]:
            problems.append(("future letter in history", e["episode_id"]))
    if any(h["text"] == e["answer"]["ntsb_letter"] for h in e["history"]):
        problems.append(("answer letter in history", e["episode_id"]))
    splits_per_case.setdefault(e["_srid"], set()).add(e["split"])
    # only check the text the model reads (day numbers like 2010 aren't years)
    v = visible(e)
    seen = " ".join([v["recommendation"]] + [h["text"] for h in v["history"]])
    if YEAR.search(seen):
        problems.append(("year visible", e["episode_id"]))
    if own_id_pattern(e["_srid"]).search(seen):
        problems.append(("real ID visible", e["episode_id"]))

problems += [("case in two splits", s) for s, sp in splits_per_case.items() if len(sp) > 1]

print(len(problems), "problems")
print(Counter(p[0] for p in problems))
print(problems[:5])

0 problems
Counter()
[]


## Step 7 · How long are the letters?
Tiny Aya only has ~8K tokens of "desk space". These numbers tell us how big a page `read_letter`
should hand out, and how many letters a typical case has. Rough rule: 1 token ≈ 4 characters.

In [12]:
def pct(values, p):
    values = sorted(values)
    return values[int(len(values) * p / 100)]

reply_chars = [len(e["history"][-1]["text"]) for e in episodes]
history_len = [len(e["history"]) for e in episodes]
answer_chars = [len(e["answer"]["ntsb_letter"]) for e in episodes]

print("reply to evaluate (chars):  median", pct(reply_chars, 50), "| 90%", pct(reply_chars, 90), "| max", max(reply_chars))
print("NTSB answer letter (chars): median", pct(answer_chars, 50), "| 90%", pct(answer_chars, 90))
print("letters in history:         median", pct(history_len, 50), "| 90%", pct(history_len, 90), "| max", max(history_len))

reply to evaluate (chars):  median 620 | 90% 1936 | max 17890
NTSB answer letter (chars): median 578 | 90% 1563
letters in history:         median 2 | 90% 7 | max 104


## Save
One JSONL file per split, saved to Drive. JSONL = one episode per line, easy to stream later.

In [13]:
for split in ["train", "val", "test"]:
    path = f"{OUT}/episodes_{split}.jsonl"
    with open(path, "w") as f:
        for e in episodes:
            if e["split"] == split:
                f.write(json.dumps(e, ensure_ascii=False) + "\n")
    print("saved", path)

!ls -lh {OUT}

saved /content/drive/MyDrive/ntsb_agent/data/episodes_train.jsonl
saved /content/drive/MyDrive/ntsb_agent/data/episodes_val.jsonl
saved /content/drive/MyDrive/ntsb_agent/data/episodes_test.jsonl
total 58M
-rw------- 1 root root 3.4M Nov 13 16:08 episodes_test.jsonl
-rw------- 1 root root  52M Nov 13 16:08 episodes_train.jsonl
-rw------- 1 root root 2.7M Nov 13 16:08 episodes_val.jsonl


In [14]:
# peek at one episode the way the model would see it (no answer, no hidden keys)
e = random.choice([e for e in episodes if e["split"] == "train" and len(e["history"]) <= 4])
print(json.dumps(visible(e), indent=2, ensure_ascii=False)[:2500])
print("\nHIDDEN ANSWER:", e["answer"]["label"])

{
  "episode_id": "CASE-3837-E1",
  "case_id": "CASE-3837",
  "mode": "Marine",
  "priority": "CLASS II",
  "most_wanted": false,
  "recipient": "",
  "recommendation": "The NTSB recommends that ergon, inc.:  distribute new name plates to all known owners of temperature probes which do not have grounding information on the nameplates.",
  "history": [
    {
      "letter_id": "L-0001",
      "day": 276,
      "days_since_previous": null,
      "from": "ORG",
      "type": "Official Correspondence",
      "text": "Your letter of July 7, [YEAR] addressed to Mr. Leslie Lampton, President of Ergon, Inc., has been forwarded to us for reply. Ergon, Inc. categorically denies that it's Thermo-Probe TP-2 portable temperature probe caused and/or contributed to the explosion aboard the Maltese Tank Vessel FIONA which occurred on August 31, [YEAR]. While we have no independent knowledge that Ergon's Thermo-Probe TP-2 was being used on the FIONA at the time of the explosion, we assume for purposes 

## Where we landed
- Episodes built from the trusted pairs, with real dates and IDs only in hidden `_` keys.
- Cleaned using the audit (`01b`): duplicates removed, undated and future-dated letters skipped,
  ALL CAPS letters turned into sentence case, Day 0 = Jan 1 of the ID year.
- Dropped: episodes whose "answer" was already in the history.
- Tagged: answers where NTSB used info that isn't in the reply (left out of letter-writing training later).
- Split by recommendation, picked on purpose: ~300 test with ~100 recent cases and every grade covered,
  ~300 validation, the rest training.
- Leak test: **0 problems**.

**Letter lengths (1 token ≈ 4 characters):**
| What | Median | 90% | Max |
|---|---|---|---|
| Reply to evaluate (chars) | 619 | 1,937 | 17,890 |
| NTSB answer letter (chars) | 577 | 1,544 | — |
| Letters in history | 3 | 7 | 165 |

A typical case fits Tiny Aya's 8K desk easily (~3,000–4,500 tokens). Settings for Phase 2:
- `read_letter` page size: **4,000 characters** — more than 90% of letters fit in one page
- `search_history`: **max 20 previews per call**, ~150 characters each
- letter-writing target: NTSB letters up to **~1,600 characters** (covers ~90%)

**Next:** the tools (Phase 2) read these files and only ever hand the model the visible part.